# EN4554 Assignment 1, Part 1: MiLoRA on a VLM

**Claim being tested (MiLoRA, Wang et al. 2025):** adapting the *minor* singular components of the pretrained weights learns the new task while leaving the knowledge in the *principal* components intact.

**Vision-language hypothesis:** fine-tuning SmolVLM-256M on TextVQA with MiLoRA keeps more zero-shot VQA ability on a held-out task (the forgetting probe) than LoRA or PiSSA, while matching them on TextVQA.

| Run | Init |
|---|---|
| Baseline | zero-shot, no adapter |
| LoRA | Kaiming A, zero B |
| PiSSA | top-r singular components |
| MiLoRA | bottom-r singular components |

Metrics: TextVQA accuracy, Δ probe accuracy (forgetting), subspace overlap of ΔW with the top-k singular vectors of W₀.

## 0. Setup

In [ ]:
# !pip install -q "transformers>=4.46" "peft>=0.13" datasets accelerate num2words

In [ ]:
import os, re, json, math, random, gc, copy
from collections import Counter
import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoProcessor, AutoModelForVision2Seq, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, PeftModel

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE, torch.__version__)

In [ ]:
# Checkpoints go to the shared Drive folder so a Colab disconnect is not fatal
IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/EN4554_A1/part1"
else:
    OUT_DIR = "./outputs/part1"
os.makedirs(OUT_DIR, exist_ok=True)

## 1. Config

In [ ]:
CFG = dict(
    model_id      = "HuggingFaceTB/SmolVLM-256M-Instruct",
    target_ds     = "lmms-lab/textvqa",          # TextVQA (train / validation)
    probe_ds      = "lmms-lab/VQAv2",            # forgetting probe; never trained on
    n_train       = 3000,
    n_val         = 500,
    n_probe       = 500,
    image_longest_edge = 512,                    # fewer image tokens -> faster on T4
    # LoRA (identical for every variant: only the init differs)
    r             = 16,
    alpha         = 16,                          # alpha == r -> scaling = 1 (makes the SVD init simple)
    dropout       = 0.0,
    # Main experiment: language-model linear layers only
    target_modules = r".*text_model.*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)",
    # Training
    lr            = 2e-4,
    epochs        = 1,
    batch_size    = 4,
    grad_accum    = 4,
    max_new_tokens = 16,
    top_k_overlap = 16,                           # k for the subspace-overlap metric
)
CFG

## 2. Model and processor

In [ ]:
processor = AutoProcessor.from_pretrained(CFG["model_id"])
processor.image_processor.size = {"longest_edge": CFG["image_longest_edge"]}

def load_base_model():
    # Load in fp32 and let Trainer's fp16 autocast handle mixed precision (T4: no bf16).
    # The SVD inits also need fp32 weights.
    m = AutoModelForVision2Seq.from_pretrained(CFG["model_id"], torch_dtype=torch.float32)
    return m.to(DEVICE)

base_model = load_base_model()
print(f"params: {sum(p.numel() for p in base_model.parameters())/1e6:.1f}M")

In [ ]:
# Inspect module names so target_modules regex is correct (vision_model / connector / text_model)
names = [n for n, m in base_model.named_modules() if isinstance(m, torch.nn.Linear)]
print(len(names)); print("\n".join(names[:5] + ["..."] + names[-5:]))

## 3. Data

In [ ]:
# Subsets only; state split and size in the report.
# streaming avoids downloading the full image datasets.
def take(ds_id, split, n, seed=SEED):
    ds = load_dataset(ds_id, split=split, streaming=True).shuffle(seed=seed, buffer_size=2000)
    return list(ds.take(n))

train_raw = take(CFG["target_ds"], "train",      CFG["n_train"])
val_raw   = take(CFG["target_ds"], "validation", CFG["n_val"])
probe_raw = take(CFG["probe_ds"],  "validation", CFG["n_probe"])
print(len(train_raw), len(val_raw), len(probe_raw))
print({k: type(v).__name__ for k, v in train_raw[0].items()})

In [ ]:
# Normalise both datasets to {image, question, answers: list[str]}.
# Field names differ between datasets; check the printout above.
def to_record(ex):
    answers = ex.get("answers") or [a["answer"] for a in ex.get("answers_list", [])]
    if answers and isinstance(answers[0], dict):
        answers = [a["answer"] for a in answers]
    img = ex["image"].convert("RGB")
    return dict(image=img, question=ex["question"], answers=answers)

train_data = [to_record(e) for e in train_raw]
val_data   = [to_record(e) for e in val_raw]
probe_data = [to_record(e) for e in probe_raw]
train_data[0]["question"], train_data[0]["answers"][:3]

In [ ]:
PROMPT_SUFFIX = " Answer briefly with a single word or short phrase."

def build_messages(question, answer=None):
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": question + PROMPT_SUFFIX}]}]
    if answer is not None:
        msgs.append({"role": "assistant", "content": [{"type": "text", "text": answer}]})
    return msgs

def majority_answer(answers):
    return Counter(answers).most_common(1)[0][0]

In [ ]:
def collate_train(batch):
    texts, images = [], []
    for ex in batch:
        msgs = build_messages(ex["question"], majority_answer(ex["answers"]))
        texts.append(processor.apply_chat_template(msgs, add_generation_prompt=False).strip())
        images.append([ex["image"]])
    enc = processor(text=texts, images=images, return_tensors="pt", padding=True)

    labels = enc["input_ids"].clone()
    # TODO (critical): loss only on the answer tokens.
    #   Mask with -100: padding, image tokens (processor.tokenizer.convert_tokens_to_ids("<image>")),
    #   and every token up to and including the "Assistant:" prefix of each sequence.
    #   Tip: tokenise the prompt-only version (add_generation_prompt=True) to get its length per example.
    raise NotImplementedError("label masking")
    enc["labels"] = labels
    return enc

# sanity check: decoded un-masked labels should be just the answer
# b = collate_train(train_data[:2]); print(processor.tokenizer.decode(b["labels"][0][b["labels"][0] != -100]))

## 4. Evaluation: VQA accuracy

In [ ]:
_contractions = {"dont": "don't", "doesnt": "doesn't", "isnt": "isn't", "cant": "can't", "wont": "won't"}
_articles = {"a", "an", "the"}
_num_map = {w: str(i) for i, w in enumerate(["zero","one","two","three","four","five","six","seven","eight","nine","ten"])}

def normalize_answer(s):
    s = s.lower().strip().replace("\n", " ").replace("\t", " ")
    s = re.sub(r"(?<=\d),(?=\d)", "", s)
    s = re.sub(r"[^\w\s']", " ", s)
    words = [_num_map.get(w, _contractions.get(w, w)) for w in s.split() if w not in _articles]
    return " ".join(words)

def vqa_accuracy(pred, gt_answers):
    # TODO (critical): official VQA metric = min(#annotators who gave `pred` / 3, 1),
    #   averaged over the 10 leave-one-out subsets of the GT answers. Normalise both sides first.
    raise NotImplementedError

In [ ]:
@torch.no_grad()
def generate_answers(model, data, batch_size=8):
    model.eval()
    processor.tokenizer.padding_side = "left"
    preds = []
    for i in tqdm(range(0, len(data), batch_size), leave=False):
        batch = data[i:i+batch_size]
        texts = [processor.apply_chat_template(build_messages(ex["question"]), add_generation_prompt=True) for ex in batch]
        enc = processor(text=texts, images=[[ex["image"]] for ex in batch], return_tensors="pt", padding=True).to(DEVICE)
        with torch.autocast("cuda", dtype=torch.float16):
            out = model.generate(**enc, max_new_tokens=CFG["max_new_tokens"], do_sample=False)
        out = out[:, enc["input_ids"].shape[1]:]
        preds += [p.strip() for p in processor.batch_decode(out, skip_special_tokens=True)]
    processor.tokenizer.padding_side = "right"
    return preds

def evaluate(model, data, name=""):
    preds = generate_answers(model, data)
    acc = float(np.mean([vqa_accuracy(p, ex["answers"]) for p, ex in zip(preds, data)])) * 100
    print(f"{name}: {acc:.2f}")
    return acc, preds

## 5. Zero-shot baseline

In [ ]:
RESULTS = {}
base_tvqa, base_tvqa_preds = evaluate(base_model, val_data, "baseline TextVQA")
base_probe, _              = evaluate(base_model, probe_data, "baseline probe")
RESULTS["baseline"] = dict(textvqa=base_tvqa, probe=base_probe)

## 6. Adapters: LoRA / PiSSA / MiLoRA

With SVD $W = U \Sigma V^\top$ and $\text{scaling}=\alpha/r$:

- **PiSSA**: $A=\sqrt{\Sigma_{[:r]}}\,V^\top_{[:r]}$, $B=U_{[:r]}\sqrt{\Sigma_{[:r]}}$, $W_{res}=W-\text{scaling}\cdot BA$ (built into `peft`).
- **MiLoRA**: the same, but using the **last** r components $[-r:]$.

In [ ]:
def base_lora_config(init="gaussian", target_modules=None):
    return LoraConfig(
        r=CFG["r"], lora_alpha=CFG["alpha"], lora_dropout=CFG["dropout"],
        target_modules=target_modules or CFG["target_modules"],
        init_lora_weights=init, bias="none",
    )

@torch.no_grad()
def milora_init_(peft_model, adapter="default"):
    from peft.tuners.lora import LoraLayer
    for name, module in peft_model.named_modules():
        if not isinstance(module, LoraLayer) or adapter not in module.lora_A:
            continue
        W = module.get_base_layer().weight              # (out, in)
        r, scaling = module.r[adapter], module.scaling[adapter]
        U, S, Vh = torch.linalg.svd(W.float(), full_matrices=False)
        # TODO (critical): take the *minor* r components, put sqrt(S/scaling) into A and B,
        #   copy into module.lora_A[adapter].weight / module.lora_B[adapter].weight,
        #   and subtract scaling * B @ A from W in place so the model output is unchanged at init.
        raise NotImplementedError("MiLoRA init")

def build_model(variant, target_modules=None):
    set_seed()
    m = load_base_model()
    if variant == "lora":
        pm = get_peft_model(m, base_lora_config("gaussian", target_modules))
    elif variant == "pissa":
        pm = get_peft_model(m, base_lora_config("pissa", target_modules))
    elif variant == "milora":
        pm = get_peft_model(m, base_lora_config(True, target_modules))
        milora_init_(pm)
    else:
        raise ValueError(variant)
    pm.print_trainable_parameters()
    return pm

In [ ]:
# Sanity check: at init every variant should produce the same logits as the base model
# (W_res + scaling*BA == W). Run on one batch before training.
def check_init_equivalence(pm, n=2):
    texts = [processor.apply_chat_template(build_messages(ex["question"]), add_generation_prompt=True) for ex in val_data[:n]]
    enc = processor(text=texts, images=[[ex["image"]] for ex in val_data[:n]], return_tensors="pt", padding=True).to(DEVICE)
    with torch.no_grad():
        a = base_model(**enc).logits; b = pm(**enc).logits
    print("max |Δlogits| =", (a - b).abs().max().item())

## 7. Claim-tied metric: subspace overlap of ΔW

For each adapted layer: $\Delta W = W_{merged} - W_0$, and
$\text{overlap}_k = \|U_k^\top \Delta W V_k\|_F \,/\, \|\Delta W\|_F$, where $U_k, V_k$ are the top-k singular vectors of $W_0$.
MiLoRA should give the lowest overlap, PiSSA the highest.

In [ ]:
@torch.no_grad()
def overlap_per_layer(pm, k=CFG["top_k_overlap"]):
    # Effective weight after training = W_res + scaling * B @ A; ΔW = that - W0 (original pretrained)
    from peft.tuners.lora import LoraLayer
    W0 = dict(base_model.named_modules())
    out = {}
    for n, m in pm.named_modules():
        if not isinstance(m, LoraLayer):
            continue
        A, B = m.lora_A["default"].weight, m.lora_B["default"].weight
        W_eff = m.get_base_layer().weight + m.scaling["default"] * (B @ A)
        W_orig = W0[n.removeprefix("base_model.model.")].weight
        out[n] = subspace_overlap(W_orig.float(), (W_eff - W_orig).float(), k)
    return out

def subspace_overlap(W0, dW, k):
    # TODO (critical): top-k left/right singular vectors of W0 -> ||U_k^T dW V_k||_F / ||dW||_F
    raise NotImplementedError

## 8. Training

In [ ]:
class ListDataset(torch.utils.data.Dataset):
    def __init__(self, data): self.data = data
    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i]

def train(pm, variant):
    args = TrainingArguments(
        output_dir=f"{OUT_DIR}/{variant}",
        num_train_epochs=CFG["epochs"],
        per_device_train_batch_size=CFG["batch_size"],
        gradient_accumulation_steps=CFG["grad_accum"],
        learning_rate=CFG["lr"], lr_scheduler_type="cosine", warmup_ratio=0.03,
        fp16=True, logging_steps=10, save_strategy="epoch", save_total_limit=1,
        remove_unused_columns=False, dataloader_pin_memory=False,
        report_to="none", seed=SEED,
    )
    trainer = Trainer(model=pm, args=args, train_dataset=ListDataset(train_data), data_collator=collate_train)
    trainer.train()
    pm.save_pretrained(f"{OUT_DIR}/{variant}/adapter")
    return trainer.state.log_history

In [ ]:
VARIANTS = ["lora", "pissa", "milora"]
LOGS, OVERLAP = {}, {}

for v in VARIANTS:
    print(f"\n===== {v} =====")
    pm = build_model(v)
    check_init_equivalence(pm)
    LOGS[v] = train(pm, v)
    tvqa, _  = evaluate(pm, val_data,   f"{v} TextVQA")
    probe, _ = evaluate(pm, probe_data, f"{v} probe")
    RESULTS[v] = dict(textvqa=tvqa, probe=probe, forgetting=probe - RESULTS["baseline"]["probe"])
    OVERLAP[v] = overlap_per_layer(pm)
    json.dump(RESULTS, open(f"{OUT_DIR}/results.json", "w"), indent=2)
    del pm; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# OVERLAP is filled inside the training loop (section 8) via overlap_per_layer(pm)
for v in VARIANTS:
    RESULTS[v]["overlap_topk"] = float(np.mean(list(OVERLAP[v].values())))
json.dump(RESULTS, open(f"{OUT_DIR}/results.json", "w"), indent=2)

## 9. Results

In [ ]:
import pandas as pd
df = pd.DataFrame(RESULTS).T[["textvqa", "probe", "forgetting", "overlap_topk"]]
df.round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
df.loc[VARIANTS, ["textvqa"]].plot.bar(ax=axes[0], legend=False, title="TextVQA acc (target)")
axes[0].axhline(RESULTS["baseline"]["textvqa"], ls="--", c="k")
df.loc[VARIANTS, ["forgetting"]].plot.bar(ax=axes[1], legend=False, title="Δ probe acc (forgetting)")
for v in VARIANTS:
    axes[2].plot(list(OVERLAP[v].values()), label=v)
axes[2].set_title(f"top-{CFG['top_k_overlap']} overlap per layer"); axes[2].legend()
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/results.png", dpi=200)

## 10. Bonus: does vision store knowledge differently from text?

1. Compare singular-value spectra (normalised, plus effective rank) of the SigLIP vision encoder, the connector and the LM weights.
2. Apply MiLoRA to each component **separately** and compare target gain vs probe forgetting.

In [ ]:
COMPONENTS = {
    "vision":    r".*vision_model.*\.(q_proj|k_proj|v_proj|out_proj|fc1|fc2)",
    "connector": r".*connector.*proj.*",
    "text":      CFG["target_modules"],
}

def effective_rank(S):
    p = S / S.sum()
    return float(torch.exp(-(p * torch.log(p + 1e-12)).sum()))

@torch.no_grad()
def spectra(model, pattern):
    out = {}
    for n, m in model.named_modules():
        if isinstance(m, torch.nn.Linear) and re.fullmatch(pattern, n):
            S = torch.linalg.svdvals(m.weight.float())
            out[n] = S.cpu()
    return out

SPECTRA = {c: spectra(base_model, p) for c, p in COMPONENTS.items()}
for c, d in SPECTRA.items():
    er = [effective_rank(S) / len(S) for S in d.values()]
    print(f"{c:9s} layers={len(d):3d}  mean normalised eff. rank={np.mean(er):.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for c, d in SPECTRA.items():
    curves = [S / S[0] for S in d.values()]
    L = min(len(s) for s in curves)
    mean = torch.stack([s[:L] for s in curves]).mean(0)
    ax.plot(np.linspace(0, 1, L), mean, label=c)
ax.set_yscale("log"); ax.set_xlabel("normalised index"); ax.set_ylabel("σ / σ_max"); ax.legend()
plt.savefig(f"{OUT_DIR}/spectra.png", dpi=200)

In [ ]:
BONUS = {}
for c in ["vision", "connector"]:   # "text" = the main MiLoRA run above
    pm = build_model("milora", target_modules=COMPONENTS[c])
    train(pm, f"milora_{c}")
    BONUS[c] = dict(textvqa=evaluate(pm, val_data, f"milora[{c}] TextVQA")[0],
                    probe=evaluate(pm, probe_data, f"milora[{c}] probe")[0])
    del pm; gc.collect(); torch.cuda.empty_cache()
BONUS["text"] = {k: RESULTS["milora"][k] for k in ("textvqa", "probe")}
pd.DataFrame(BONUS).T

## 11. Findings (fill in for the report)

- Does MiLoRA match LoRA / PiSSA on TextVQA?
- Does MiLoRA forget less on the probe?
- Does the overlap metric line up with the forgetting ranking?
- Bonus: does the spectrum differ between vision and text, and does that predict where MiLoRA helps?